# Notebook 01: Train to Deploy

# Outline 

1. Import Libraries and Functions
2. Import Raw Data
3. Clean Raw Data
4. Configure Model
5. Train Model 
6. Save Artifacts

# 1. Import Libraries and Functions

In [0]:
import os
import sys

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import pandas as pd

# sections 2, 3 and 6
from classifier import (
    ID_LEVELS, NAME_LEVELS, TEXT_COL, ORIG_TEXT_COL, DELIMITER,
    raw_data_path, cleaned_data_path, model_dir, timestamp,
)

# section 3 — installs nltk and fetches the word list on first use
from classifier.cleaning import build_cleaner, clean_training_data
# section 4
from classifier import ModelSettings
# section 5
from classifier.train import train
# section 6
from classifier import artifacts

In [0]:
# One stamp per run, shared by both data files and the model directory.
STAMP = timestamp()
print("run stamp:", STAMP)

run stamp: 20260921_204647


# 2. Import Raw Data

In [0]:
# %sql
# with 
#     base as (
#         select s.id as `ID`, 
#                s.erp_cono as `Cono`, 
#                s.erp_product_number as `Vallen ID`, 
#                replace(i.descrip, ';', ' ') as `FullDesc`, 
#                s.category_l4_id
#         from prd_bronze.stibo_ftp.sku s 
#         inner join prd_bronze.openedge.icsp i 
#         on s.erp_product_number = i.prod
#         where s.st_row_current=1 
#             and s.category_l4_id is not null 
#             and i.st_row_current=1 
#             and i.cono in (1,55) 
#             and i.descrip is not null
#     ), 

#     hierarchy as (
#         select
#             l1.id   AS `Level 1 ID`,
#             l1.name AS `Level 1 Name`,
#             l2.id   AS `Level 2 ID`,
#             l2.name AS `Level 2 Name`,
#             l3.id   AS `Level 3 ID`,
#             l3.name AS `Level 3 Name`,
#             l4.id   AS `Level 4 ID`,
#             l4.name AS `Level 4 Name`
#         from prd_bronze.stibo_ftp.hierarchy l1
#         left join prd_bronze.stibo_ftp.hierarchy l2
#             on l2.parent_id = l1.id
#             and l2.level = 2
#             and l2.st_row_current = 1
#         left join prd_bronze.stibo_ftp.hierarchy l3
#             on l3.parent_id = l2.id
#             and l3.level = 3
#             and l3.st_row_current = 1
#         left join prd_bronze.stibo_ftp.hierarchy l4
#             on l4.parent_id = l3.id
#             and l4.level = 4
#             and l4.st_row_current = 1
#         where l1.level = 1
#             and l1.st_row_current = 1
#     )

# select *
# from base b
# inner join hierarchy h
# on b.category_l4_id = h.`Level 4 ID` 
# where h.`Level 4 ID` is not null

In [0]:
# df = _sqldf.toPandas()
# print(f"{len(df):,} rows, {df.shape[1]} columns")

In [0]:
# # optional: filter data (remove test set rows)
# # ============================================
# import pandas as pd
# to_pred = pd.read_csv(r"data/to_predict.txt", 
#                       delimiter='|', 
#                       dtype='str')

# print(len(df))
# df = df[df['ID'].isin(to_pred['<ID>']) == False]
# print(len(df))

In [0]:
# df.to_csv(raw_data_path(STAMP),
#           index=False,
#           sep=DELIMITER)
# print("wrote", raw_data_path(STAMP))

# 3. Clean Data

In [0]:
cleaner = build_cleaner()
print(cleaner.describe())

[bootstrap] installing nltk>=3.8
[bootstrap] downloading the 'words' corpus
[bootstrap] 234,377 English words available
DescriptionCleaner(lists=['color_abbrevs', 'colors', 'sizes', 'unit_abbrevs', 'unit_plurals', 'units'], vocabulary=234,377 words)


In [0]:
# cleaned = clean_training_data(
#     df, cleaner, text_col=TEXT_COL, orig_col=ORIG_TEXT_COL,
# )
# cleaned[[ORIG_TEXT_COL, TEXT_COL]].head(10)

In [0]:
# def remove_tokens_from_text(df, text_column, tokens_csv_path, token_column=None, case_sensitive=False):
#     # Load tokens from CSV (assumes single-column, no header; adjust token_column if it has one)
#     if token_column is not None:
#         tokens_df = pd.read_csv(tokens_csv_path)
#         tokens = tokens_df[token_column].dropna().astype(str).tolist()
#     else:
#         tokens_df = pd.read_csv(tokens_csv_path, header=None)
#         tokens = tokens_df.iloc[:, 0].dropna().astype(str).tolist()

#     keep_set = set(tokens) if case_sensitive else {t.lower() for t in tokens}

#     def clean(text):
#         if pd.isna(text):
#             return text
#         words = str(text).split()  # splits on whitespace runs; also strips leading/trailing whitespace
#         if case_sensitive:
#             filtered = [w for w in words if w in keep_set]
#         else:
#             filtered = [w for w in words if w.lower() in keep_set]
#         return ' '.join(filtered)  # rejoins with single spaces

#     df = df.copy()
#     df[text_column] = df[text_column].apply(clean)
#     df = df[df[text_column].notna() & (df[text_column] != '')]  # 4. drop now-empty rows
#     df = df.reset_index(drop=True)

#     return df

In [0]:
# cleaned = remove_tokens_from_text(cleaned, 'FullDesc', r'data/good_tokens.csv')

In [0]:
# import re

# _WS = re.compile(r"\s+")
# _SIMPLE = re.compile(r"^[A-Za-z0-9]+$")       # a term the fast path can handle


# def build_replacer(pairs, case_insensitive=True, flexible_whitespace=True):
#     """
#     pairs : DataFrame with 'find' and 'replace' columns,
#             or any iterable of (find, replace) tuples.

#     Returns a function replace(text) -> text.
#     """
#     if isinstance(pairs, pd.DataFrame):
#         pairs = pairs[["find", "replace"]].itertuples(index=False, name=None)

#     lookup = {}          # normalised term -> replacement
#     complex_terms = []   # terms containing spaces or punctuation

#     for find, repl in pairs:
#         if not isinstance(find, str) or not find.strip():
#             continue
#         find = _WS.sub(" ", find.strip())
#         repl = "" if repl is None or pd.isna(repl) else str(repl).strip()

#         key = find.casefold() if case_insensitive else find
#         if key in lookup:                      # first spelling of a dupe wins
#             continue
#         lookup[key] = repl
#         if not _SIMPLE.match(find):
#             complex_terms.append(find)

#     if not lookup:
#         raise ValueError("no usable rows in the find/replace table")

#     # Longest first so 'END MILL' is tried before 'END', and 'Q-TIP' before 'Q'.
#     complex_terms.sort(key=len, reverse=True)

#     def as_pattern(term):
#         p = re.escape(term)
#         if flexible_whitespace:
#             p = p.replace(r"\ ", r"\s+")       # tolerate double spaces, newlines
#         return p

#     # Two branches, tried left to right:
#     #   1. the punctuated / multi-word terms, fenced by lookarounds. Lookarounds
#     #      rather than \b, because \b misbehaves for terms that begin or end
#     #      with punctuation (e.g. '"', '(', '%').
#     #   2. a bare alphanumeric run, which is maximal-munch, so a term can never
#     #      match only part of a longer word ('MILL' inside 'MILLING').
#     branches = []
#     if complex_terms:
#         branches.append(
#             r"(?<![A-Za-z0-9])(?:"
#             + "|".join(as_pattern(t) for t in complex_terms)
#             + r")(?![A-Za-z0-9])"
#         )
#     branches.append(r"[A-Za-z0-9]+")

#     pattern = re.compile("|".join(branches),
#                          re.IGNORECASE if case_insensitive else 0)

#     def substitute(m):
#         text = m.group(0)
#         key = _WS.sub(" ", text)
#         if case_insensitive:
#             key = key.casefold()
#         return lookup.get(key, text)          # not in the table -> leave as-is

#     def replace(text):
#         if not isinstance(text, str):
#             return text                        # NaN / None / numbers pass through
#         return pattern.sub(substitute, text)

#     return replace

In [0]:
# pairs = pd.read_csv(r"data/to_normalize.csv")          # columns: find, replace
# replace = build_replacer(pairs)
# cleaned["FullDesc"] = cleaned["FullDesc"].map(replace)

In [0]:
# cleaned.to_csv(cleaned_data_path(STAMP),
#                index=False,
#                sep=DELIMITER)
# print("wrote", cleaned_data_path(STAMP))

In [0]:
cleaned = pd.read_csv(r"data/cleaned_data.txt", 
                      delimiter='|', 
                      dtype='str')

# 4. Configure Model

In [0]:
settings = ModelSettings()

# Every value the model depends on is stated here. Nothing is left to a
# library default, including the parameters sklearn would otherwise fill
# in silently.

settings.vectorizer.max_features = 125_000
settings.vectorizer.min_df = 2   # int = document count
settings.min_path_count = 5
settings.vectorizer.ngram_range = (1, 2)

# Vectorizer Params — both kinds
settings.vectorizer.kind = "tfidf"
settings.vectorizer.max_df = 0.95           # float = proportion
settings.vectorizer.stop_words = "english"

settings.vectorizer.lowercase = True
settings.vectorizer.token_pattern = r"(?u)\b\w\w+\b"   # drops 1-char tokens
settings.vectorizer.binary = False          # 04 set True, but the old
                                            # build() passed binary to
                                            # CountVectorizer only, so
                                            # the model 04 trained had
                                            # binary=False

# Vectorizer Params — tfidf only
settings.vectorizer.sublinear_tf = True     # 1 + log(tf)
settings.vectorizer.norm = "l2"             # each row to unit length
settings.vectorizer.use_idf = False         # no inverse-document weighting
settings.vectorizer.smooth_idf = True       # inert while use_idf is False

# Memory-Related Params
settings.batch_size = 1000
settings.fit_chunk_size = 20000
settings.score_dtype = "float64"

# Model Hyperparams
#   joint_path        one model per level, marginals multiplied
#   conditional_path  one model per parent, chain rule. Its scores sum to
#                     one across valid paths, unlike joint_path, so the
#                     thresholds in config/thresholds.json do not carry
#                     between the two modes.
settings.prediction_mode = "conditional_path"

# conditional_path only. A parent whose subtree holds fewer rows than
# this gets no model and its children are treated as equally likely.
# 0 models every parent that has at least two children.
settings.min_node_rows = 0

# conditional_path only. False skips the per-level models -- that is
# where the memory saving is -- but leaves joint_path unavailable.
settings.fit_flat_levels = True
settings.nb.fit_prior = True
settings.nb.alpha = 0.001
settings.nb.class_prior = None

settings

ModelSettings(min_path_count=5, batch_size=1000, fit_chunk_size=20000, prediction_mode='conditional_path', min_node_rows=0, fit_flat_levels=True, score_dtype='float64', vectorizer=VectorizerSettings(kind='tfidf', ngram_range=(1, 2), min_df=2, max_df=0.95, stop_words='english', max_features=125000, lowercase=True, token_pattern='(?u)\\b\\w\\w+\\b', binary=False, sublinear_tf=True, norm='l2', use_idf=False, smooth_idf=True), nb=NBSettings(alpha=0.001, fit_prior=True, class_prior=None))

# 5. Train Model

In [0]:
# free up some memory
# del df

In [0]:
clf, trained_on = train(cleaned, settings)

[filter]   585,858  rows in
[filter]   585,690  after dropping null text or labels (168 dropped)
[filter]   583,418  after dropping paths seen < 5 times (2,272 dropped, 4,645 -> 3,623 categories)
[train] fitting on all 583,418 rows (no holdout)
[fit] vectorized 583,418 rows into 125,000 features (sparse, nnz=3,980,954) in 6.0s
[fit] trained 'Level 1 ID': 23 classes in 1.9s
[fit] trained 'Level 2 ID': 155 classes in 13.9s
[fit] trained 'Level 3 ID': 727 classes in 76.9s
[fit] trained 'Level 4 ID': 3,623 classes in 427.6s
[fit] 3,623 paths observed in training vs. 9,389,928,365 possible combinations
[fit] depth 1: 1 parent model(s), 0 left uniform, 23 stacked columns in 2.6s
[fit] depth 2: 22 parent model(s), 1 left uniform, 154 stacked columns in 4.6s
[fit] depth 3: 113 parent model(s), 42 left uniform, 685 stacked columns in 6.3s
[fit] depth 4: 549 parent model(s), 178 left uniform, 3,445 stacked columns in 13.1s


# 6. Save Model

In [0]:
MODEL_DIR = model_dir(STAMP)

written = artifacts.save(
    clf,
    MODEL_DIR,
    cleaner=cleaner,
    hierarchy_lookup=trained_on[ID_LEVELS + NAME_LEVELS],
    n_training_rows=len(trained_on),
)
for name, path in written.items():
    print(f"{name:<24} {os.path.basename(path)}")

vectorizer               vectorizer.pkl
settings                 settings.pkl
cleaner                  cleaner.pkl
path_index               path_index.pkl
model::Level 1 ID        model_Level_1_ID.pkl
model::Level 2 ID        model_Level_2_ID.pkl
model::Level 3 ID        model_Level_3_ID.pkl
model::Level 4 ID        model_Level_4_ID.pkl
node_stack               node_stack.pkl
hierarchy_lookup         hierarchy_lookup.csv
manifest                 manifest.json


In [0]:
# Confirm the saved copy reproduces the model in memory before relying on it.
reloaded, reloaded_cleaner, lookup, manifest = artifacts.load(MODEL_DIR)

_sample = trained_on[TEXT_COL].head(300)
_before = clf.predict(_sample)
_after = reloaded.predict(_sample)

_same = (_before[ID_LEVELS].values == _after[ID_LEVELS].values).all()
# Relative, not absolute: these scores run from about 1e-6 down to 1e-48,
# so a fixed 1e-12 tolerance would wave through almost any discrepancy.
_a, _b = _before["L4 Confidence"], _after["L4 Confidence"]
_close = ((_a - _b).abs() / _a.abs().clip(lower=1e-300)).max() < 1e-12
print(f"identical predictions: {bool(_same)} | identical confidence: {bool(_close)}")
assert _same and _close, "the saved model does not reproduce the fitted one"

print(f"\nmodel saved to {MODEL_DIR}")
print("notebook 02 picks up the most recent model_* directory automatically.")

---------------------------------------------------------------------------
The Python process exited with exit code 137 (SIGKILL: Killed). This may have been caused by an OOM error. Check your command's memory usage.



The last 10 KB of the process's stderr and stdout can be found below. See driver logs for full logs.
---------------------------------------------------------------------------
Last messages on stderr:
---------------------------------------------------------------------------
Last messages on stdout:
1
run stamp: 20260921_204647
[bootstrap] installing nltk>=3.8
[bootstrap] downloading the 'words' corpus
[bootstrap] 234,377 English words available
DescriptionCleaner(lists=['color_abbrevs', 'colors', 'sizes', 'unit_abbrevs', 'unit_plurals', 'units'], vocabulary=234,377 words)
[filter]   585,858  rows in
[filter]   585,690  after dropping null text or labels (168 dropped)
[filter]   583,418  after dropping paths seen < 5 times (2,272 dropped, 4,645 -> 3,623 categories)
[